# SRGAN MODEL IMPLEMENTATION

We will be developing all 9 models we defined in model_desing.

## Complete Experimental Training Design

| Model | Scenario | Downsampling Method | Blur Type | Noise Type | Scale Factor | LR Size |
|-------|----------|---------------------|-----------|------------|--------------|---------|
| Model 1 | A | Bicubic | Gaussian ($\sigma = 1.0$) | Gaussian ($\sigma_n = 5$) | ×2 | 256×256 |
| Model 2 | A | Bicubic | Gaussian ($\sigma = 1.0$) | Gaussian ($\sigma_n = 5$) | ×4 | 128×128 |
| Model 3 | A | Bicubic | Gaussian ($\sigma = 1.0$) | Gaussian ($\sigma_n = 5$) | ×8 | 64×64 |
| Model 4 | B | Bilinear | Gaussian ($\sigma = 1.5$) | Gaussian ($\sigma_n = 10$) | ×2 | 256×256 |
| Model 5 | B | Bilinear | Gaussian ($\sigma = 1.5$) | Gaussian ($\sigma_n = 10$) | ×4 | 128×128 |
| Model 6 | B | Bilinear | Gaussian ($\sigma = 1.5$) | Gaussian ($\sigma_n = 10$) | ×8 | 64×64 |
| Model 7 | C | Lanczos | Gaussian ($\sigma = 2.0$) | Poisson | ×2 | 256×256 |
| Model 8 | C | Lanczos | Gaussian ($\sigma = 2.0$) | Poisson | ×4 | 128×128 |
| Model 9 | C | Lanczos | Gaussian ($\sigma = 2.0$) | Poisson | ×8 | 64×64 |
| Model 1 | A | Bicubic | Gaussian ($\sigma = 1.0$) | Gaussian ($\sigma_n = 5$) | ×2 | 256×256 |
| Model 4 | B | Bilinear | Gaussian ($\sigma = 1.5$) | Gaussian ($\sigma_n = 10$) | ×2 | 256×256 |
| Model 7 | C | Lanczos | Gaussian ($\sigma = 2.0$) | Poisson | ×2 | 256×256 |

In [32]:
import torch
import sys

print("=" * 60)
print("DIAGNÓSTICO DE CUDA")
print("=" * 60)
print(f"Python executable: {sys.executable}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
    print(f"Compute capability: {torch.cuda.get_device_capability(0)}")
else:
    print("\n⚠️  CUDA NO ESTÁ DISPONIBLE")
    print("   Verifica que:")
    print("   1. Tienes GPU NVIDIA instalada")
    print("   2. NVIDIA drivers están actualizados")
    print("   3. PyTorch se instaló con CUDA support")
print("=" * 60)

DIAGNÓSTICO DE CUDA
Python executable: e:\Proyects Python based\ProyectoAvanzado2\venv311\Scripts\python.exe
PyTorch version: 2.1.0+cu118
CUDA disponible: True
GPU: NVIDIA GeForce GTX 1650 with Max-Q Design
CUDA version: 11.8
Compute capability: (7, 5)


In [33]:
import os
import json
import random
import warnings
from pathlib import Path
from dataclasses import dataclass, field
from typing import Literal

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
import torchvision.models as models
import cv2
from PIL import Image
import matplotlib.pyplot as plt
from skimage.metrics import peak_signal_noise_ratio as psnr_sk
from skimage.metrics import structural_similarity as ssim_sk
import lpips
from tqdm import tqdm

warnings.filterwarnings("ignore")

# ─── Reproducibilidad global ──────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ─── Dispositivo ──────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Entrenando en: {DEVICE}")

# ─── Rutas ────────────────────────────────────────────────────────────────────
DATA_DIR      = Path(r"E:\Proyects Python based\ProyectoAvanzado2\data\processed\M3_diagnosis\images")          # ← CAMBIA ESTA RUTA a tu carpeta de imágenes
RESULTS_DIR   = Path(r"E:\Proyects Python based\ProyectoAvanzado2\models\Super-resolution\results")
CHECKPOINTS_DIR = Path(r"E:\Proyects Python based\ProyectoAvanzado2\models\Super-resolution\checkpoints")
METRICS_DIR   = Path(r"E:\Proyects Python based\ProyectoAvanzado2\models\Super-resolution\metrics")

for d in [RESULTS_DIR, CHECKPOINTS_DIR, METRICS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

Entrenando en: cuda


## 2. Tabla de Experimentos

Definimos los 9 modelos experimentales como objetos Python tipados. Cada `ExperimentConfig`
encapsula completamente un escenario: método de degradación, tipo de ruido, factor de escala
y tamaño de imagen LR (baja resolución). Esta arquitectura permite iterar los 9 modelos
en un loop sin duplicar código.

Los tres escenarios representan niveles crecientes de dificultad:
- **Escenario A**: degradación suave (bicubic + Gaussian σ=1.0 + ruido σ=5)
- **Escenario B**: degradación media (bilinear + Gaussian σ=1.5 + ruido σ=10)
- **Escenario C**: degradación severa (Lanczos + Gaussian σ=2.0 + ruido Poisson)

In [34]:
@dataclass
class ExperimentConfig:
    model_id: int
    scenario: Literal["A", "B", "C"]
    downsample_method: Literal["bicubic", "bilinear", "lanczos"]
    blur_sigma: float          # σ del filtro Gaussiano de desenfoque
    noise_type: Literal["gaussian", "poisson"]
    noise_sigma: float         # σ_n para ruido Gaussiano (ignorado si Poisson)
    scale_factor: int          # ×2, ×4 o ×8
    lr_size: int               # Tamaño del parche LR (256, 128, 64)

    @property
    def hr_size(self) -> int:
        return self.lr_size * self.scale_factor

    @property
    def name(self) -> str:
        return f"model_{self.model_id}_scenario{self.scenario}_x{self.scale_factor}"


EXPERIMENTS = [
    # ── Escenario A: Bicubic ──────────────────────────────────────────────────
    ExperimentConfig(1, "A", "bicubic",  1.0, "gaussian", 5,  2, 256),
    ExperimentConfig(2, "A", "bicubic",  1.0, "gaussian", 5,  4, 128),
    ExperimentConfig(3, "A", "bicubic",  1.0, "gaussian", 5,  8,  64),
    # ── Escenario B: Bilinear ─────────────────────────────────────────────────
    ExperimentConfig(4, "B", "bilinear", 1.5, "gaussian", 10, 2, 256),
    ExperimentConfig(5, "B", "bilinear", 1.5, "gaussian", 10, 4, 128),
    ExperimentConfig(6, "B", "bilinear", 1.5, "gaussian", 10, 8,  64),
    # ── Escenario C: Lanczos ──────────────────────────────────────────────────
    ExperimentConfig(7, "C", "lanczos",  2.0, "poisson",  0,  2, 256),
    ExperimentConfig(8, "C", "lanczos",  2.0, "poisson",  0,  4, 128),
    ExperimentConfig(9, "C", "lanczos",  2.0, "poisson",  0,  8,  64),
]

# Visualizar tabla de configuraciones
print(f"{'ID':<6} {'Escenario':<10} {'Downsample':<12} {'Blur σ':<8} {'Ruido':<12} {'Escala':<8} {'LR':<8} {'HR':<8}")
print("─" * 75)
for e in EXPERIMENTS:
    noise_str = f"Gaussian σ={e.noise_sigma}" if e.noise_type == "gaussian" else "Poisson"
    print(f"M{e.model_id:<5} {e.scenario:<10} {e.downsample_method:<12} {e.blur_sigma:<8} {noise_str:<12} ×{e.scale_factor:<7} {e.lr_size}px   {e.hr_size}px")

ID     Escenario  Downsample   Blur σ   Ruido        Escala   LR       HR      
───────────────────────────────────────────────────────────────────────────
M1     A          bicubic      1.0      Gaussian σ=5 ×2       256px   512px
M2     A          bicubic      1.0      Gaussian σ=5 ×4       128px   512px
M3     A          bicubic      1.0      Gaussian σ=5 ×8       64px   512px
M4     B          bilinear     1.5      Gaussian σ=10 ×2       256px   512px
M5     B          bilinear     1.5      Gaussian σ=10 ×4       128px   512px
M6     B          bilinear     1.5      Gaussian σ=10 ×8       64px   512px
M7     C          lanczos      2.0      Poisson      ×2       256px   512px
M8     C          lanczos      2.0      Poisson      ×4       128px   512px
M9     C          lanczos      2.0      Poisson      ×8       64px   512px


## 3. Pipeline de Degradación (HR → LR)

El corazón de los experimentos es la función de degradación. Dado que entrenamos con pares
(LR, HR) sintéticos, debemos simular de forma controlada el proceso de degradación que
se daría en imágenes reales capturadas con equipos de baja resolución.

El pipeline sigue este orden: **Blur → Downsample → Ruido**. Este orden es físicamente
correcto: en una cámara real primero el sistema óptico desenfoca (PSF), luego el sensor
muestrea a menor resolución, y finalmente el sensor añade ruido electrónico/fotónico.

In [35]:
def gaussian_blur(img: np.ndarray, sigma: float) -> np.ndarray:
    """Aplica desenfoque Gaussiano. kernel_size se calcula como 6σ+1 (cubre ±3σ)."""
    ksize = int(6 * sigma + 1)
    if ksize % 2 == 0:
        ksize += 1
    return cv2.GaussianBlur(img, (ksize, ksize), sigma)


def downsample(img: np.ndarray, scale: int, method: str) -> np.ndarray:
    """Reduce resolución con el método especificado por la configuración experimental."""
    h, w = img.shape[:2]
    target_h, target_w = h // scale, w // scale
    interp_map = {
        "bicubic":  cv2.INTER_CUBIC,
        "bilinear": cv2.INTER_LINEAR,
        "lanczos":  cv2.INTER_LANCZOS4,
    }
    return cv2.resize(img, (target_w, target_h), interpolation=interp_map[method])


def add_noise(img: np.ndarray, noise_type: str, sigma: float) -> np.ndarray:
    """
    Añade ruido a la imagen LR.
    - Gaussiano: ruido aditivo N(0, σ_n²), simula ruido electrónico del sensor.
    - Poisson: ruido multiplicativo proporcional a la intensidad, simula ruido fotónico.
    """
    img_float = img.astype(np.float32) / 255.0

    if noise_type == "gaussian":
        noise = np.random.normal(0, sigma / 255.0, img_float.shape).astype(np.float32)
        noisy = img_float + noise

    elif noise_type == "poisson":
        # Escalar para que Poisson tenga efecto visible; vals típicos: 50-200
        scale_val = 100.0
        noisy = np.random.poisson(img_float * scale_val).astype(np.float32) / scale_val

    return np.clip(noisy * 255.0, 0, 255).astype(np.uint8)


def degrade_image(hr_img: np.ndarray, cfg: ExperimentConfig) -> np.ndarray:
    """Pipeline completo de degradación: Blur → Downsample → Ruido."""
    blurred = gaussian_blur(hr_img, cfg.blur_sigma)
    lr = downsample(blurred, cfg.scale_factor, cfg.downsample_method)
    lr_noisy = add_noise(lr, cfg.noise_type, cfg.noise_sigma)
    return lr_noisy


# ─── Verificación visual del pipeline ────────────────────────────────────────
def visualize_degradation(image_path: str, configs: list, n_show: int = 3):
    """Muestra la degradación para los primeros n_show experimentos."""
    hr = cv2.imread(image_path)
    hr = cv2.cvtColor(hr, cv2.COLOR_BGR2RGB)
    # Recortar un patch HR central para la demo
    h, w = hr.shape[:2]

    fig, axes = plt.subplots(1, n_show + 1, figsize=(4 * (n_show + 1), 4))
    axes[0].imshow(hr); axes[0].set_title("HR Original"); axes[0].axis("off")

    for i, cfg in enumerate(configs[:n_show]):
        # HR debe ser múltiplo de scale_factor
        hr_crop_size = cfg.hr_size
        cy, cx = h // 2, w // 2
        hr_patch = hr[cy - hr_crop_size//2 : cy + hr_crop_size//2,
                      cx - hr_crop_size//2 : cx + hr_crop_size//2]
        lr_patch = degrade_image(hr_patch, cfg)
        axes[i+1].imshow(lr_patch)
        axes[i+1].set_title(f"M{cfg.model_id}: ×{cfg.scale_factor}\n{cfg.downsample_method}")
        axes[i+1].axis("off")

    plt.suptitle("Pipeline de Degradación HR → LR", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "degradation_preview.png", dpi=150)
    plt.show()

# Ejemplo de uso — reemplaza con una imagen de tu dataset:
# visualize_degradation("./dataset/imagen_001.jpg", EXPERIMENTS)

## 4. Dataset Personalizado (SRDataset)

Implementamos un `Dataset` de PyTorch que genera pares (LR, HR) on-the-fly. Esta decisión
es importante para datasets de ~3500 imágenes: generar los pares al vuelo evita almacenar
las 9 versiones degradadas de cada imagen en disco (lo que consumiría varios GBs extra).

El recorte aleatorio de patches (en lugar de usar la imagen completa) cumple dos funciones:
1. **Aumenta** artificialmente el número de muestras de entrenamiento.
2. **Normaliza** el tamaño de entrada, necesario para batching en PyTorch.

In [36]:
class SRDataset(Dataset):
    """
    Dataset de super-resolución que genera pares (LR, HR) on-the-fly.
    
    Args:
        image_paths: lista de rutas a imágenes HR de alta resolución.
        cfg: configuración experimental (define degradación y tamaños).
        augment: si True, aplica volteos y rotaciones aleatorias.
    """
    def __init__(self, image_paths: list, cfg: ExperimentConfig, augment: bool = True):
        self.image_paths = image_paths
        self.cfg = cfg
        self.augment = augment

    def __len__(self) -> int:
        return len(self.image_paths)

    def __getitem__(self, idx: int):
        # ── Cargar imagen HR ──────────────────────────────────────────────────
        img = cv2.imread(str(self.image_paths[idx]))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]

        # ── Recorte aleatorio de patch HR ─────────────────────────────────────
        hr_size = self.cfg.hr_size
        if h < hr_size or w < hr_size:
            img = cv2.resize(img, (max(w, hr_size), max(h, hr_size)), cv2.INTER_CUBIC)
            h, w = img.shape[:2]

        top  = random.randint(0, h - hr_size)
        left = random.randint(0, w - hr_size)
        hr_patch = img[top:top+hr_size, left:left+hr_size]

        # ── Augmentación ──────────────────────────────────────────────────────
        if self.augment:
            if random.random() > 0.5:
                hr_patch = np.fliplr(hr_patch).copy()
            if random.random() > 0.5:
                hr_patch = np.flipud(hr_patch).copy()
            k = random.choice([0, 1, 2, 3])
            hr_patch = np.rot90(hr_patch, k).copy()

        # ── Degradar HR → LR ──────────────────────────────────────────────────
        lr_patch = degrade_image(hr_patch, self.cfg)

        # ── Convertir a tensores normalizados [-1, 1] (estándar SRGAN) ────────
        hr_tensor = torch.from_numpy(hr_patch).permute(2, 0, 1).float() / 127.5 - 1.0
        lr_tensor = torch.from_numpy(lr_patch).permute(2, 0, 1).float() / 127.5 - 1.0

        return lr_tensor, hr_tensor


def build_dataloaders(
    image_paths: list,
    cfg,
    val_split: float = 0.1,
    batch_size: int = 16,
    num_workers: int = 0  # 🔥 FORZADO a 0 (Windows safe)
):
    """
    Divide el dataset en train/val y retorna DataLoaders.
    val_split=0.1 → 10% validación, 90% entrenamiento.
    """

    print("\n🔧 Building DataLoaders...")

    # ================= SPLIT =================
    n_val = max(1, int(len(image_paths) * val_split))
    indices = list(range(len(image_paths)))
    random.shuffle(indices)

    val_paths   = [image_paths[i] for i in indices[:n_val]]
    train_paths = [image_paths[i] for i in indices[n_val:]]

    print(f"📊 Total imágenes: {len(image_paths)}")
    print(f"📊 Train: {len(train_paths)} | Val: {len(val_paths)}")

    # ================= DATASETS =================
    train_ds = SRDataset(train_paths, cfg, augment=True)
    val_ds   = SRDataset(val_paths,   cfg, augment=False)

    # ================= DEVICE CONFIG =================
    pin_memory = torch.cuda.is_available()

    print(f"⚙️ Config:")
    print(f"   Batch size: {batch_size}")
    print(f"   Num workers: {num_workers} (Windows safe mode)")
    print(f"   Pin memory: {pin_memory}")

    # ================= DATALOADERS =================
    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,   # 🔥 CLAVE
        pin_memory=pin_memory
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=1,
        shuffle=False,
        num_workers=0,             # 🔥 SIEMPRE 0 en validación
        pin_memory=pin_memory
    )

    print(f"📦 Batches por época: {len(train_loader)}")

    return train_loader, val_loader

## 5. Arquitectura SRGAN

Implementamos SRGAN según el paper original (Ledig et al., 2017) con las siguientes
componentes:

**Generator (SRResNet + upsampling):**
- Bloque inicial Conv → LeakyReLU
- 16 Bloques Residuales (ResBlocks) con skip connections (preservan gradiente profundo)
- Sub-pixel convolution (PixelShuffle) para upsampling eficiente (aprende el upsample)
- El número de bloques PixelShuffle se determina automáticamente según el scale_factor

**Discriminator:**
- Red convolucional profunda estilo VGG con BatchNorm
- Clasifica si un patch es real (HR) o generado (SR)
- Salida escalar (probabilidad real/falso)

**Por qué PixelShuffle en lugar de Transposed Convolution:**
PixelShuffle evita el artefacto "checkerboard" (tablero de ajedrez) común en
transposed convolutions, produciendo upsampling más uniforme y nítido.

In [37]:
# ──────────────────────────────────────────────────────────────────────────────
# 5.1  Bloque Residual del Generator
# ──────────────────────────────────────────────────────────────────────────────
class ResBlock(nn.Module):
    def __init__(self, channels: int = 64):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(channels, channels, 3, 1, 1),
            nn.BatchNorm2d(channels),
            nn.PReLU(),
            nn.Conv2d(channels, channels, 3, 1, 1),
            nn.BatchNorm2d(channels),
        )

    def forward(self, x):
        return x + self.block(x)   # skip connection


# ──────────────────────────────────────────────────────────────────────────────
# 5.2  Generator
# ──────────────────────────────────────────────────────────────────────────────
class Generator(nn.Module):
    """
    Generator SRGAN. Soporta scale_factor ∈ {2, 4, 8} automáticamente.
    - scale=2  → 1 PixelShuffle(×2)
    - scale=4  → 2 PixelShuffle(×2)
    - scale=8  → 3 PixelShuffle(×2)
    """
    def __init__(self, scale_factor: int, n_res_blocks: int = 16, channels: int = 64):
        super().__init__()
        assert scale_factor in (2, 4, 8), "scale_factor debe ser 2, 4 u 8"

        # Entrada
        self.head = nn.Sequential(
            nn.Conv2d(3, channels, 9, 1, 4),
            nn.PReLU()
        )

        # Bloques residuales
        self.res_blocks = nn.Sequential(*[ResBlock(channels) for _ in range(n_res_blocks)])

        # Post-residual
        self.post_res = nn.Sequential(
            nn.Conv2d(channels, channels, 3, 1, 1),
            nn.BatchNorm2d(channels),
        )

        # Upsampling: stack de bloques PixelShuffle(×2)
        n_ups = int(np.log2(scale_factor))   # 1, 2 o 3
        ups_layers = []
        for _ in range(n_ups):
            ups_layers += [
                nn.Conv2d(channels, channels * 4, 3, 1, 1),
                nn.PixelShuffle(2),           # channels*4 → channels, H*2, W*2
                nn.PReLU()
            ]
        self.upsample = nn.Sequential(*ups_layers)

        # Salida
        self.tail = nn.Conv2d(channels, 3, 9, 1, 4)
        self.tanh = nn.Tanh()

    def forward(self, x):
        head_out = self.head(x)
        res_out  = self.post_res(self.res_blocks(head_out))
        x_up     = self.upsample(head_out + res_out)   # skip global
        return self.tanh(self.tail(x_up))


# ──────────────────────────────────────────────────────────────────────────────
# 5.3  Discriminator
# ──────────────────────────────────────────────────────────────────────────────
def _disc_block(in_ch, out_ch, stride):
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, 3, stride, 1, bias=False),
        nn.BatchNorm2d(out_ch),
        nn.LeakyReLU(0.2, inplace=True),
    )

class Discriminator(nn.Module):
    def __init__(self, hr_size: int):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, 3, 1, 1),
            nn.LeakyReLU(0.2, inplace=True),
            _disc_block( 64,  64, 2),
            _disc_block( 64, 128, 1),
            _disc_block(128, 128, 2),
            _disc_block(128, 256, 1),
            _disc_block(256, 256, 2),
            _disc_block(256, 512, 1),
            _disc_block(512, 512, 2),
        )
        # Calcular tamaño de salida de features para la capa densa
        feat_size = hr_size // 16   # 4 strided convs de stride=2
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(feat_size),
            nn.Flatten(),
            nn.Linear(512 * feat_size * feat_size, 1024),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(1024, 1),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


# ──────────────────────────────────────────────────────────────────────────────
# Verificación de parámetros
# ──────────────────────────────────────────────────────────────────────────────
for scale in [2, 4, 8]:
    g = Generator(scale)
    n_params = sum(p.numel() for p in g.parameters()) / 1e6
    print(f"Generator ×{scale}: {n_params:.2f}M parámetros")

Generator ×2: 1.40M parámetros
Generator ×4: 1.55M parámetros
Generator ×8: 1.70M parámetros


## 6. Funciones de Pérdida SRGAN

SRGAN combina tres pérdidas, cada una con un propósito distinto:

**1. Pérdida de Contenido (VGG Perceptual Loss):**
Compara las activaciones de la capa `relu3_3` de VGG-19 entre la imagen SR generada
y la HR real. Esto fuerza al generator a producir imágenes perceptualmente similares,
no solo pixel-a-pixel. Es la pérdida más importante para calidad visual.

**2. Pérdida Adversarial del Generator:**
El generator es penalizado cuando el discriminador lo "detecta" como falso. Esto
empuja al generator a sintetizar texturas realistas que engañen al discriminador.

**3. Pérdida de Píxeles (L1):**
Pérdida auxiliar que estabiliza el entrenamiento inicial. Tiende a producir imágenes
algo suavizadas pero asegura fidelidad básica en luminancia y estructura global.

**Función total del Generator:**
`L_G = λ_content · L_VGG + λ_adv · L_adv + λ_pixel · L_pixel`
con `λ_content=1.0, λ_adv=1e-3, λ_pixel=1e-2` (valores del paper).

In [38]:
class VGGPerceptualLoss(nn.Module):
    """
    Pérdida perceptual usando activaciones de VGG-19 (relu3_3 = capa 18).
    VGG se congela completamente — solo se usa como extractor de features.
    """
    def __init__(self, layer_idx: int = 18):
        super().__init__()
        vgg = models.vgg19(weights=models.VGG19_Weights.IMAGENET1K_V1)
        self.feature_extractor = nn.Sequential(*list(vgg.features)[:layer_idx]).eval()
        for param in self.feature_extractor.parameters():
            param.requires_grad = False
        self.criterion = nn.MSELoss()

        # Normalización ImageNet (VGG fue entrenado con estas stats)
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1,3,1,1))
        self.register_buffer("std",  torch.tensor([0.229, 0.224, 0.225]).view(1,3,1,1))

    def forward(self, sr: torch.Tensor, hr: torch.Tensor) -> torch.Tensor:
        # Desnormalizar de [-1,1] → [0,1] → normalizar ImageNet
        sr_n = ((sr + 1) / 2 - self.mean) / self.std
        hr_n = ((hr + 1) / 2 - self.mean) / self.std
        return self.criterion(self.feature_extractor(sr_n),
                              self.feature_extractor(hr_n))


class SRGANLoss(nn.Module):
    """Pérdida total del Generator con pesos configurables."""
    def __init__(self, lambda_content=1.0, lambda_adv=1e-3, lambda_pixel=1e-2):
        super().__init__()
        self.vgg_loss   = VGGPerceptualLoss()
        self.adv_crit   = nn.BCEWithLogitsLoss()
        self.pixel_crit = nn.L1Loss()
        self.lc = lambda_content
        self.la = lambda_adv
        self.lp = lambda_pixel

    def forward(self, sr, hr, disc_sr_logits):
        l_content = self.vgg_loss(sr, hr)
        real_labels = torch.ones_like(disc_sr_logits)
        l_adv   = self.adv_crit(disc_sr_logits, real_labels)
        l_pixel = self.pixel_crit(sr, hr)
        total = self.lc * l_content + self.la * l_adv + self.lp * l_pixel
        return total, {"content": l_content.item(),
                       "adversarial": l_adv.item(),
                       "pixel": l_pixel.item()}

## 7. Bucle de Entrenamiento

El entrenamiento SRGAN sigue el protocolo de dos fases:

**Fase 1 — Pre-entrenamiento del Generator (SRResNet):**
Entrenamos el generator solo con pérdida L1 durante algunas épocas. Esto inicializa
los pesos en una solución razonable antes de introducir el discriminador. Sin esta fase,
el adversarial training desde cero puede ser inestable (mode collapse).

**Fase 2 — Entrenamiento GAN completo:**
Alternamos las actualizaciones de Discriminador y Generator:
- Paso D: el discriminador aprende a distinguir HR real de SR falso.
- Paso G: el generator aprende a engañar al discriminador (pérdida adversarial + VGG + L1).

Se guardan checkpoints del mejor modelo según PSNR en validación.

In [39]:
def pretrain_generator(generator, train_loader, n_epochs: int, cfg: ExperimentConfig):
    """
    Fase 1: Pre-entrenamiento del Generator con pérdida L1.
    Inicializa pesos en una buena solución antes del adversarial training.
    """
    generator.train()
    optimizer = optim.Adam(generator.parameters(), lr=1e-4, betas=(0.9, 0.999))
    criterion = nn.L1Loss()

    print(f"\n{'─'*60}")
    print(f"  PRE-ENTRENAMIENTO Generator — {cfg.name}")
    print(f"{'─'*60}")

    for epoch in range(1, n_epochs + 1):
        total_loss = 0.0
        pbar = tqdm(train_loader, desc=f"Pre-train Epoch {epoch}/{n_epochs}")
        for lr_batch, hr_batch in pbar:
            lr_batch = lr_batch.to(DEVICE)
            hr_batch = hr_batch.to(DEVICE)

            sr_batch = generator(lr_batch)
            loss = criterion(sr_batch, hr_batch)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            pbar.set_postfix({"L1": f"{loss.item():.4f}"})

        print(f"  Epoch {epoch:03d} | L1_avg: {total_loss/len(train_loader):.5f}")

    return generator, optimizer


def train_srgan(cfg: ExperimentConfig, image_paths: list,
                n_pretrain: int = 10, n_epochs: int = 100,
                batch_size: int = 16, lr_g: float = 1e-4, lr_d: float = 1e-4):
    """
    Entrenamiento completo SRGAN para una configuración experimental.
    
    Returns:
        generator: modelo entrenado (mejor según PSNR de validación)
        history: diccionario con métricas por época
    """
    generator = Generator(cfg.scale_factor).to(DEVICE)

    print(f"🧠 Generator device: {next(generator.parameters()).device}")
    # ── Dataloaders ───────────────────────────────────────────────────────────
    train_loader, val_loader = build_dataloaders(image_paths, cfg, batch_size=batch_size)

    # ── Modelos ───────────────────────────────────────────────────────────────
    generator     = Generator(cfg.scale_factor).to(DEVICE)
    discriminator = Discriminator(cfg.hr_size).to(DEVICE)
    srgan_loss    = SRGANLoss().to(DEVICE)
    disc_crit     = nn.BCEWithLogitsLoss()

    # ── Fase 1: Pre-entrenamiento ─────────────────────────────────────────────
    generator, _ = pretrain_generator(generator, train_loader, n_pretrain, cfg)

    # ── Optimizadores (Fase 2) ────────────────────────────────────────────────
    opt_G = optim.Adam(generator.parameters(),     lr=lr_g, betas=(0.9, 0.999))
    opt_D = optim.Adam(discriminator.parameters(), lr=lr_d, betas=(0.9, 0.999))

    # LR scheduler: reduce ×10 a la mitad del entrenamiento (protocolo paper)
    scheduler_G = optim.lr_scheduler.StepLR(opt_G, step_size=n_epochs//2, gamma=0.1)
    scheduler_D = optim.lr_scheduler.StepLR(opt_D, step_size=n_epochs//2, gamma=0.1)

    history = {"loss_G": [], "loss_D": [], "psnr_val": [], "ssim_val": []}
    best_psnr = -1.0
    ckpt_path = CHECKPOINTS_DIR / f"{cfg.name}_best.pth"

    print(f"\n{'═'*60}")
    print(f"  ENTRENAMIENTO GAN — {cfg.name}")
    print(f"  HR: {cfg.hr_size}×{cfg.hr_size} | LR: {cfg.lr_size}×{cfg.lr_size}")
    print(f"{'═'*60}")

    for epoch in range(1, n_epochs + 1):
        generator.train(); discriminator.train()
        ep_loss_G, ep_loss_D = 0.0, 0.0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{n_epochs}")
        for lr_batch, hr_batch in pbar:
            lr_batch = lr_batch.to(DEVICE)
            hr_batch = hr_batch.to(DEVICE)
            bs = lr_batch.size(0)

            # ── Paso Discriminador ────────────────────────────────────────────
            with torch.no_grad():
                sr_batch = generator(lr_batch)

            real_logits = discriminator(hr_batch)
            fake_logits = discriminator(sr_batch.detach())

            loss_D_real = disc_crit(real_logits, torch.ones(bs, 1).to(DEVICE))
            loss_D_fake = disc_crit(fake_logits, torch.zeros(bs, 1).to(DEVICE))
            loss_D = (loss_D_real + loss_D_fake) / 2

            opt_D.zero_grad(); loss_D.backward(); opt_D.step()

            # ── Paso Generator ────────────────────────────────────────────────
            sr_batch = generator(lr_batch)
            fake_logits_G = discriminator(sr_batch)
            loss_G, _ = srgan_loss(sr_batch, hr_batch, fake_logits_G)

            opt_G.zero_grad(); loss_G.backward(); opt_G.step()

            ep_loss_G += loss_G.item()
            ep_loss_D += loss_D.item()
            pbar.set_postfix({"G": f"{loss_G.item():.4f}", "D": f"{loss_D.item():.4f}"})

        scheduler_G.step(); scheduler_D.step()

        # ── Validación ────────────────────────────────────────────────────────
        if epoch % 5 == 0 or epoch == n_epochs:
            val_psnr, val_ssim = evaluate(generator, val_loader)
            history["psnr_val"].append(val_psnr)
            history["ssim_val"].append(val_ssim)
            print(f"  → Epoch {epoch:03d} | PSNR: {val_psnr:.3f} dB | SSIM: {val_ssim:.4f}")

            # Guardar mejor checkpoint
            if val_psnr > best_psnr:
                best_psnr = val_psnr
                torch.save({"epoch": epoch,
                            "generator_state": generator.state_dict(),
                            "psnr": val_psnr,
                            "cfg": cfg.name}, ckpt_path)
                print(f"  ✓ Checkpoint guardado (PSNR: {best_psnr:.3f} dB)")

        history["loss_G"].append(ep_loss_G / len(train_loader))
        history["loss_D"].append(ep_loss_D / len(train_loader))

    # Cargar mejor checkpoint al terminar
    best_ckpt = torch.load(ckpt_path, map_location=DEVICE)
    generator.load_state_dict(best_ckpt["generator_state"])
    print(f"\n  Mejor PSNR alcanzado: {best_psnr:.3f} dB (época {best_ckpt['epoch']})")
    return generator, history

## 8. Métricas de Evaluación

Evaluamos cada modelo con cuatro métricas complementarias:

| Métrica | Rango | Mejor valor | Qué mide |
|---------|-------|-------------|----------|
| **PSNR** | 0–∞ dB | Más alto | Fidelidad de píxeles (relación señal/ruido) |
| **SSIM** | 0–1 | → 1 | Similitud estructural (luminancia, contraste, estructura) |
| **LPIPS** | 0–∞ | → 0 | Similitud perceptual (basada en features de VGG) |
| **MSE** | 0–∞ | → 0 | Error cuadrático medio (pixel-level) |

**PSNR y SSIM** son las métricas clásicas de reconstrucción.
**LPIPS** (Zhang et al., 2018) correlaciona mejor con la percepción humana y es
especialmente relevante para evaluar GANs, que pueden tener PSNR moderado pero
alta calidad perceptual.

In [40]:
def tensor_to_uint8(t: torch.Tensor) -> np.ndarray:
    """Convierte tensor [-1,1] → numpy uint8 [0,255]."""
    arr = ((t.clamp(-1, 1) + 1) / 2 * 255).byte().cpu().numpy()
    return arr.transpose(1, 2, 0)   # C,H,W → H,W,C


@torch.no_grad()
def evaluate(generator: nn.Module, val_loader: DataLoader,
             lpips_fn=None) -> tuple:
    """
    Calcula PSNR y SSIM promedio sobre el conjunto de validación.
    lpips_fn opcional para evaluación completa (más lenta).
    """
    generator.eval()
    psnr_list, ssim_list = [], []

    for lr_batch, hr_batch in val_loader:
        lr_batch = lr_batch.to(DEVICE)
        sr_batch = generator(lr_batch)

        for sr, hr in zip(sr_batch, hr_batch):
            sr_np = tensor_to_uint8(sr)
            hr_np = tensor_to_uint8(hr)

            psnr_val = psnr_sk(hr_np, sr_np, data_range=255)
            ssim_val = ssim_sk(hr_np, sr_np, data_range=255, channel_axis=2)
            psnr_list.append(psnr_val)
            ssim_list.append(ssim_val)

    return np.mean(psnr_list), np.mean(ssim_list)


@torch.no_grad()
def full_evaluation(generator: nn.Module, val_loader: DataLoader,
                    cfg: ExperimentConfig) -> dict:
    """
    Evaluación completa con PSNR, SSIM, LPIPS y MSE.
    Se ejecuta una sola vez al final del entrenamiento.
    """
    generator.eval()
    lpips_fn = lpips.LPIPS(net="vgg").to(DEVICE)

    metrics = {"psnr": [], "ssim": [], "lpips": [], "mse": []}

    for lr_batch, hr_batch in tqdm(val_loader, desc=f"Evaluando {cfg.name}"):
        lr_batch = lr_batch.to(DEVICE)
        hr_batch_dev = hr_batch.to(DEVICE)
        sr_batch = generator(lr_batch)

        # LPIPS opera en tensores [-1,1] directamente
        lp_val = lpips_fn(sr_batch, hr_batch_dev).mean().item()

        for sr, hr in zip(sr_batch, hr_batch):
            sr_np = tensor_to_uint8(sr)
            hr_np = tensor_to_uint8(hr)

            metrics["psnr"].append(psnr_sk(hr_np, sr_np, data_range=255))
            metrics["ssim"].append(ssim_sk(hr_np, sr_np, data_range=255, channel_axis=2))
            metrics["mse"].append(np.mean((sr_np.astype(float) - hr_np.astype(float))**2))

        metrics["lpips"].append(lp_val)

    return {k: float(np.mean(v)) for k, v in metrics.items()}

## 9. Ejecución Completa — Los 9 Modelos

Este bloque es el orquestador principal. Itera sobre los 9 experimentos, entrena cada
modelo, evalúa con métricas completas y guarda los resultados en un JSON consolidado.

**Estimación de tiempo:** con GPU (RTX 3080/4090) cada modelo toma ~2-4 horas
dependiendo del scale_factor. Sin GPU (CPU), se recomienda reducir `n_epochs` y
`batch_size` drásticamente para pruebas, y usar GPU para el experimento real.

**Manejo de fallos:** si un modelo falla, el loop continúa con el siguiente y
registra el error. Así no se pierde trabajo ya completado.

In [41]:
from pathlib import Path

def load_image_paths(data_dir, extensions=(".jpg", ".jpeg", ".png", ".bmp", ".tif")):
    print("🔍 Escaneando imágenes...")
    
    data_dir = Path(data_dir)
    paths = []

    for ext in extensions:
        paths.extend(data_dir.rglob(f"*{ext}"))
        paths.extend(data_dir.rglob(f"*{ext.upper()}"))

    paths = sorted(set(paths))

    print(f"✅ {len(paths)} imágenes encontradas")
    return paths

import time
from pathlib import Path

# ================= FUNCIÓN =================
def load_image_paths(data_dir, extensions=(".jpg", ".jpeg", ".png", ".bmp", ".tif")):
    print("🔍 Escaneando imágenes...")
    
    data_dir = Path(data_dir)
    paths = []

    for ext in extensions:
        paths.extend(data_dir.rglob(f"*{ext}"))
        paths.extend(data_dir.rglob(f"*{ext.upper()}"))

    paths = sorted(set(paths))

    print(f"✅ {len(paths)} imágenes encontradas")
    return paths


# ================= STEP 1 =================
print("\n🔍 STEP 1: Loading image paths...")
start = time.time()

image_paths = load_image_paths(
    r"E:\Proyects Python based\ProyectoAvanzado2\data\processed\M3_diagnosis\images"
)

print(f"⏱️ Time loading paths: {time.time() - start:.2f} sec")

assert len(image_paths) > 0, "No images found"


# 🔥 DEBUG MODE
image_paths = image_paths[:50]
print(f"⚡ Debug mode: using {len(image_paths)} images")


# ================= LOOP =================
for cfg in EXPERIMENTS[:1]:

    print(f"\n{'█'*60}")
    print(f"  DEBUGGING: {cfg.name.upper()}")
    print(f"{'█'*60}")

    try:
        # ================= STEP 2 =================
        print("\n🔍 STEP 2: Building DataLoader...")
        start = time.time()

        train_loader, val_loader = build_dataloaders(
            image_paths,
            cfg,
            batch_size=4
        )

        print(f"⏱️ Time building loaders: {time.time() - start:.2f} sec")

        # ================= STEP 3 =================
        print("\n🔍 STEP 3: Testing Dataset access...")
        dataset = train_loader.dataset

        for i in range(5):
            t0 = time.time()
            lr, hr = dataset[i]
            print(f"✅ Sample {i} loaded in {time.time() - t0:.3f} sec")
            print(f"   LR shape: {lr.shape} | HR shape: {hr.shape}")

        # ================= STEP 4 =================
        print("\n🔍 STEP 4: Testing DataLoader iteration...")
        start = time.time()

        for i, (lr_batch, hr_batch) in enumerate(train_loader):
            print(f"\n✅ Batch {i} loaded")
            print(f"   LR batch: {lr_batch.shape}")
            print(f"   HR batch: {hr_batch.shape}")
            print(f"   ⏱️ Batch load time: {time.time() - start:.2f} sec")
            break

        print("🎉 DataLoader is working correctly")

    except Exception as e:
        print(f"\n❌ CRITICAL ERROR: {e}")

    break


🔍 STEP 1: Loading image paths...
🔍 Escaneando imágenes...
✅ 3578 imágenes encontradas
⏱️ Time loading paths: 0.27 sec
⚡ Debug mode: using 50 images

████████████████████████████████████████████████████████████
  DEBUGGING: MODEL_1_SCENARIOA_X2
████████████████████████████████████████████████████████████

🔍 STEP 2: Building DataLoader...

🔧 Building DataLoaders...
📊 Total imágenes: 50
📊 Train: 45 | Val: 5
⚙️ Config:
   Batch size: 4
   Num workers: 0 (Windows safe mode)
   Pin memory: True
📦 Batches por época: 12
⏱️ Time building loaders: 0.00 sec

🔍 STEP 3: Testing Dataset access...
✅ Sample 0 loaded in 0.023 sec
   LR shape: torch.Size([3, 256, 256]) | HR shape: torch.Size([3, 512, 512])
✅ Sample 1 loaded in 0.021 sec
   LR shape: torch.Size([3, 256, 256]) | HR shape: torch.Size([3, 512, 512])
✅ Sample 2 loaded in 0.020 sec
   LR shape: torch.Size([3, 256, 256]) | HR shape: torch.Size([3, 512, 512])
✅ Sample 3 loaded in 0.020 sec
   LR shape: torch.Size([3, 256, 256]) | HR shape: tor

In [42]:
import json
import torch
import gc
import time
from pathlib import Path

# ================= CARGAR DATASET =================
print("\n🔍 Cargando rutas de imágenes...")
start = time.time()

image_paths = load_image_paths(DATA_DIR)

print(f"⏱️ Tiempo carga: {time.time() - start:.2f} sec")
assert len(image_paths) > 0, f"No se encontraron imágenes en {DATA_DIR}"

# ================= CONFIGURACIÓN =================
# 🔥 AJUSTADO PARA GPU 4GB
TRAIN_CFG = {
    "n_pretrain":  5,     # 🔽 menos para debug/estabilidad
    "n_epochs":   20,     # 🔽 evita tiempos eternos
    "batch_size":  2,     # 🔥 CLAVE para evitar OOM
    "lr_g":       1e-4,
    "lr_d":       1e-4,
}

print("\n⚙️ CONFIGURACIÓN:")
for k, v in TRAIN_CFG.items():
    print(f"   {k}: {v}")

# ================= LOOP EXPERIMENTOS =================
all_results = {}

for cfg in EXPERIMENTS:

    print(f"\n{'█'*60}")
    print(f"  INICIANDO: {cfg.name.upper()}")
    print(f"  Escenario {cfg.scenario} | {cfg.downsample_method} | ×{cfg.scale_factor}")
    print(f"{'█'*60}")

    try:
        # 🔥 LIMPIAR GPU ANTES DE EMPEZAR
        gc.collect()
        torch.cuda.empty_cache()

        if torch.cuda.is_available():
            print(f"🧠 GPU antes: {torch.cuda.memory_allocated()/1024**2:.2f} MB")

        # ── ENTRENAMIENTO ─────────────────────────
        start_train = time.time()

        generator, history = train_srgan(
            cfg=cfg,
            image_paths=image_paths,
            **TRAIN_CFG
        )

        print(f"⏱️ Tiempo entrenamiento: {(time.time() - start_train)/60:.2f} min")

        # 🔥 LIMPIEZA PARCIAL ANTES DE EVALUAR
        torch.cuda.empty_cache()

        # ── EVALUACIÓN FINAL ─────────────────────
        print("\n🔍 Evaluando modelo...")

        _, val_loader = build_dataloaders(
            image_paths,
            cfg,
            batch_size=1   # 🔥 evaluación ligera
        )

        final_metrics = full_evaluation(generator, val_loader, cfg)

        # ── GUARDAR RESULTADOS ───────────────────
        final_metrics["model_id"] = cfg.model_id
        final_metrics["scenario"] = cfg.scenario
        final_metrics["scale_factor"] = cfg.scale_factor
        final_metrics["downsample"] = cfg.downsample_method

        all_results[cfg.name] = {
            "metrics": final_metrics,
            "history": {
                k: [round(v, 6) for v in vals]
                for k, vals in history.items()
            }
        }

        print(f"\n  ✅ {cfg.name} completado:")
        print(f"     PSNR:  {final_metrics['psnr']:.3f} dB")
        print(f"     SSIM:  {final_metrics['ssim']:.4f}")
        print(f"     LPIPS: {final_metrics['lpips']:.4f}")
        print(f"     MSE:   {final_metrics['mse']:.2f}")

        # ── LIBERAR GPU COMPLETAMENTE ────────────
        del generator
        gc.collect()
        torch.cuda.empty_cache()

        if torch.cuda.is_available():
            print(f"🧠 GPU después: {torch.cuda.memory_allocated()/1024**2:.2f} MB")

    except RuntimeError as e:
        print(f"\n  ❌ Error en {cfg.name}: {e}")

        # 🔥 Si es OOM → limpiar fuerte
        if "out of memory" in str(e).lower():
            print("⚠️ Liberando memoria GPU...")
            gc.collect()
            torch.cuda.empty_cache()

        all_results[cfg.name] = {"error": str(e)}

    except Exception as e:
        print(f"\n  ❌ Error inesperado en {cfg.name}: {e}")
        all_results[cfg.name] = {"error": str(e)}

# ================= GUARDAR RESULTADOS =================
results_path = METRICS_DIR / "all_results.json"

with open(results_path, "w") as f:
    json.dump(all_results, f, indent=2)

print(f"\n✅ Resultados guardados en: {results_path}")


🔍 Cargando rutas de imágenes...
🔍 Escaneando imágenes...
✅ 3578 imágenes encontradas
⏱️ Tiempo carga: 0.14 sec

⚙️ CONFIGURACIÓN:
   n_pretrain: 5
   n_epochs: 20
   batch_size: 2
   lr_g: 0.0001
   lr_d: 0.0001

████████████████████████████████████████████████████████████
  INICIANDO: MODEL_1_SCENARIOA_X2
  Escenario A | bicubic | ×2
████████████████████████████████████████████████████████████
🧠 GPU antes: 0.00 MB
🧠 Generator device: cuda:0

🔧 Building DataLoaders...
📊 Total imágenes: 3578
📊 Train: 3221 | Val: 357
⚙️ Config:
   Batch size: 2
   Num workers: 0 (Windows safe mode)
   Pin memory: True
📦 Batches por época: 1611

────────────────────────────────────────────────────────────
  PRE-ENTRENAMIENTO Generator — model_1_scenarioA_x2
────────────────────────────────────────────────────────────


Pre-train Epoch 1/5:   1%|▏         | 24/1611 [01:57<2:09:34,  4.90s/it, L1=0.1037]


KeyboardInterrupt: 

## 10. Análisis Comparativo de los 9 Modelos

Con todos los modelos entrenados, procedemos al análisis comparativo. Visualizamos:
1. **Tabla resumen** de métricas por modelo y escenario.
2. **Curvas de entrenamiento** (loss G/D, PSNR por época).
3. **Heatmaps** de métricas agrupadas por escenario × scale_factor.
4. **Comparativa visual** LR vs SR vs HR para cada configuración.

In [ ]:
def load_results(path: Path = METRICS_DIR / "all_results.json") -> dict:
    with open(path) as f:
        return json.load(f)


def print_metrics_table(results: dict):
    """Imprime tabla resumen de métricas en consola."""
    header = f"{'Modelo':<35} {'PSNR':>8} {'SSIM':>8} {'LPIPS':>8} {'MSE':>10}"
    print("\n" + "─"*70)
    print(header)
    print("─"*70)
    for name, data in results.items():
        if "error" in data:
            print(f"{name:<35} {'ERROR':>8}")
            continue
        m = data["metrics"]
        print(f"{name:<35} {m['psnr']:>8.3f} {m['ssim']:>8.4f} {m['lpips']:>8.4f} {m['mse']:>10.2f}")
    print("─"*70)


def plot_training_curves(results: dict):
    """Curvas de loss y PSNR de validación para todos los modelos."""
    fig, axes = plt.subplots(3, 3, figsize=(18, 14))
    axes = axes.flatten()

    for i, (name, data) in enumerate(results.items()):
        if "error" in data or i >= 9:
            continue
        ax = axes[i]
        h = data["history"]
        epochs = range(1, len(h["loss_G"]) + 1)

        ax2 = ax.twinx()
        ax.plot(epochs, h["loss_G"], "b-", lw=1.5, label="Loss G", alpha=0.8)
        ax.plot(epochs, h["loss_D"], "r-", lw=1.5, label="Loss D", alpha=0.8)

        psnr_epochs = range(5, len(h["psnr_val"]) * 5 + 1, 5)
        ax2.plot(list(psnr_epochs)[:len(h["psnr_val"])], h["psnr_val"],
                 "g--", lw=2, label="PSNR val", marker="o", markersize=4)

        ax.set_title(name.replace("model_", "M").replace("_scenario", " | Esc."),
                     fontsize=9, fontweight="bold")
        ax.set_xlabel("Época"); ax.set_ylabel("Loss", color="blue")
        ax2.set_ylabel("PSNR (dB)", color="green")
        ax.legend(loc="upper left", fontsize=7)
        ax2.legend(loc="upper right", fontsize=7)

    plt.suptitle("Curvas de Entrenamiento — 9 Modelos SRGAN", fontsize=16, fontweight="bold")
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "training_curves.png", dpi=150)
    plt.show()


def plot_metrics_heatmap(results: dict):
    """Heatmap de PSNR y SSIM: filas=Escenario, columnas=Scale Factor."""
    scenarios  = ["A", "B", "C"]
    scales     = [2, 4, 8]
    psnr_grid  = np.zeros((3, 3))
    ssim_grid  = np.zeros((3, 3))
    lpips_grid = np.zeros((3, 3))

    for name, data in results.items():
        if "error" in data:
            continue
        m = data["metrics"]
        s_idx = scenarios.index(m["scenario"])
        c_idx = scales.index(m["scale_factor"])
        psnr_grid[s_idx, c_idx]  = m["psnr"]
        ssim_grid[s_idx, c_idx]  = m["ssim"]
        lpips_grid[s_idx, c_idx] = m["lpips"]

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for ax, grid, title, cmap in zip(axes,
                                      [psnr_grid, ssim_grid, lpips_grid],
                                      ["PSNR (dB) ↑", "SSIM ↑", "LPIPS ↓"],
                                      ["RdYlGn", "RdYlGn", "RdYlGn_r"]):
        im = ax.imshow(grid, cmap=cmap, aspect="auto")
        ax.set_xticks([0, 1, 2]); ax.set_xticklabels(["×2", "×4", "×8"])
        ax.set_yticks([0, 1, 2]); ax.set_yticklabels(["A (Bicubic)", "B (Bilinear)", "C (Lanczos)"])
        ax.set_title(title, fontsize=14, fontweight="bold")
        plt.colorbar(im, ax=ax)
        for i in range(3):
            for j in range(3):
                ax.text(j, i, f"{grid[i,j]:.3f}", ha="center", va="center",
                        fontsize=12, fontweight="bold", color="black")

    plt.suptitle("Comparativa de Métricas por Escenario y Factor de Escala",
                 fontsize=16, fontweight="bold")
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "metrics_heatmap.png", dpi=150)
    plt.show()


def visual_comparison(generator: nn.Module, image_path: str, cfg: ExperimentConfig):
    """Muestra LR | SR (SRGAN) | HR para una imagen de prueba."""
    hr = cv2.imread(image_path)
    hr = cv2.cvtColor(hr, cv2.COLOR_BGR2RGB)
    h, w = hr.shape[:2]

    # Patch central
    hs = cfg.hr_size
    cy, cx = h//2, w//2
    hr_patch = hr[cy-hs//2:cy+hs//2, cx-hs//2:cx+hs//2]
    lr_patch = degrade_image(hr_patch, cfg)

    # Inferencia
    lr_t = torch.from_numpy(lr_patch).permute(2,0,1).float().unsqueeze(0) / 127.5 - 1
    generator.eval()
    with torch.no_grad():
        sr_t = generator(lr_t.to(DEVICE))
    sr_np = tensor_to_uint8(sr_t.squeeze(0))

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for ax, img, title in zip(axes,
                               [lr_patch, sr_np, hr_patch],
                               [f"LR ({cfg.lr_size}px)", f"SR×{cfg.scale_factor} (SRGAN)",
                                f"HR ({cfg.hr_size}px)"]):
        ax.imshow(img); ax.set_title(title, fontsize=13, fontweight="bold"); ax.axis("off")

    psnr_val = psnr_sk(hr_patch, sr_np, data_range=255)
    ssim_val = ssim_sk(hr_patch, sr_np, data_range=255, channel_axis=2)
    plt.suptitle(f"{cfg.name} | PSNR: {psnr_val:.2f} dB | SSIM: {ssim_val:.4f}",
                 fontsize=14)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / f"visual_{cfg.name}.png", dpi=150)
    plt.show()


# ─── Ejecutar análisis ────────────────────────────────────────────────────────
results = load_results()
print_metrics_table(results)
plot_training_curves(results)
plot_metrics_heatmap(results)

FileNotFoundError: [Errno 2] No such file or directory: 'E:\\Proyects Python based\\ProyectoAvanzado2\\models\\Super-resolution\\metrics\\all_results.json'

## 11. Inferencia: Aplicar Modelo Entrenado a Nuevas Imágenes

Una vez entrenados los 9 modelos, podemos aplicar cualquiera de ellos a imágenes
nuevas (no vistas durante entrenamiento). Esta función carga el checkpoint del mejor
modelo y procesa la imagen completa en patches solapados para evitar artefactos en bordes.

In [ ]:
def load_best_generator(cfg: ExperimentConfig) -> nn.Module:
    """Carga el generator con los mejores pesos guardados durante entrenamiento."""
    ckpt_path = CHECKPOINTS_DIR / f"{cfg.name}_best.pth"
    if not ckpt_path.exists():
        raise FileNotFoundError(f"No se encontró checkpoint: {ckpt_path}")

    generator = Generator(cfg.scale_factor).to(DEVICE)
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    generator.load_state_dict(ckpt["generator_state"])
    generator.eval()
    print(f"✓ Checkpoint cargado: {cfg.name} (PSNR entrenamiento: {ckpt.get('psnr', 'N/A'):.3f} dB)")
    return generator


@torch.no_grad()
def super_resolve_image(generator: nn.Module, image_path: str,
                        cfg: ExperimentConfig, patch_size: int = 128,
                        overlap: int = 16) -> np.ndarray:
    """
    Aplica super-resolución a una imagen completa usando procesamiento por patches.
    El overlap suaviza las costuras entre patches adyacentes.
    
    Args:
        patch_size: tamaño del patch LR de entrada
        overlap: solapamiento en píxeles entre patches (reduce artefactos de borde)
    """
    lr_img = cv2.imread(image_path)
    lr_img = cv2.cvtColor(lr_img, cv2.COLOR_BGR2RGB)
    h_lr, w_lr = lr_img.shape[:2]

    # Canvas de salida SR
    h_sr = h_lr * cfg.scale_factor
    w_sr = w_lr * cfg.scale_factor
    sr_canvas = np.zeros((h_sr, w_sr, 3), dtype=np.float32)
    weight_map = np.zeros((h_sr, w_sr, 1), dtype=np.float32)
    step = patch_size - overlap

    for y in range(0, h_lr, step):
        for x in range(0, w_lr, step):
            y_end = min(y + patch_size, h_lr)
            x_end = min(x + patch_size, w_lr)
            patch = lr_img[y:y_end, x:x_end]

            # Padding si el patch es más pequeño
            pad_h = patch_size - patch.shape[0]
            pad_w = patch_size - patch.shape[1]
            if pad_h > 0 or pad_w > 0:
                patch = np.pad(patch, ((0, pad_h), (0, pad_w), (0, 0)), mode="reflect")

            t = torch.from_numpy(patch).permute(2,0,1).float().unsqueeze(0) / 127.5 - 1
            sr_patch = generator(t.to(DEVICE)).squeeze(0)
            sr_patch_np = ((sr_patch.clamp(-1,1) + 1) / 2 * 255).cpu().numpy().transpose(1,2,0)

            # Colocar patch en canvas SR (coordenadas escaladas)
            y_sr, x_sr = y * cfg.scale_factor, x * cfg.scale_factor
            ph_sr = (y_end - y) * cfg.scale_factor
            pw_sr = (x_end - x) * cfg.scale_factor

            sr_canvas[y_sr:y_sr+ph_sr, x_sr:x_sr+pw_sr] += sr_patch_np[:ph_sr, :pw_sr]
            weight_map[y_sr:y_sr+ph_sr, x_sr:x_sr+pw_sr] += 1.0

    sr_final = np.clip(sr_canvas / weight_map, 0, 255).astype(np.uint8)
    return sr_final


# ─── Ejemplo de uso ───────────────────────────────────────────────────────────
# cfg_demo = EXPERIMENTS[0]   # Modelo 1: Escenario A, ×2
# gen = load_best_generator(cfg_demo)
# sr_result = super_resolve_image(gen, "./nueva_imagen.jpg", cfg_demo)
# Image.fromarray(sr_result).save(RESULTS_DIR / "sr_output_model1.png")
# print("✅ Imagen super-resuelta guardada.")

## 12. Reporte Final en CSV

Exportamos la tabla de métricas consolidada en formato CSV para análisis posterior
en herramientas como Excel, R o cualquier pipeline estadístico externo.

In [ ]:
import csv

def export_csv_report(results: dict, path: Path = METRICS_DIR / "report_final.csv"):
    """Exporta métricas de todos los modelos a CSV."""
    fieldnames = ["model_id", "scenario", "downsample", "scale_factor",
                  "psnr", "ssim", "lpips", "mse"]

    with open(path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for name, data in sorted(results.items()):
            if "error" in data:
                continue
            m = data["metrics"]
            writer.writerow({k: m.get(k, "") for k in fieldnames})

    print(f"✅ Reporte CSV exportado: {path}")

    # Preview
    import pandas as pd
    df = pd.read_csv(path)
    print(df.to_string(index=False))


export_csv_report(results)